In [1]:
import os
import pandas as pd

In [2]:
# 解析DockQ输出为DataFrame（支持method从model_path中提取）
def parse_dockq_output(out_file):
    """从合并的DockQ输出文件解析结果，自动从model_path提取method"""
    
    if not os.path.exists(out_file):
        raise FileNotFoundError(f"DockQ output file not found: {out_file}")
    
    rows = []
    with open(out_file, 'r') as f:
        lines = f.readlines()
    
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 20:
            continue  # 跳过 "Total DockQ over..." 等非数据行
        try:
            dockq_score = float(parts[1])
            irmsd = float(parts[3])
            lrmsd = float(parts[5])
            fnat = float(parts[7])
            
            # 从model_path中提取method、pdb、seed、sample
            model_path = parts[16]
            native_path = parts[20]
            
            # path: .../original_result/{method}/output/{pdb}/seed-{seed}-sample-{sample}/{pdb}_seed-{seed}_sample-{sample}_model.cif
            # 提取method: 找到 /original_result/ 后的第一个目录名
            method = model_path.split('/PepSet/')[1].split('/')[0]
        
            native_name = native_path.split('/')[-1].replace('.pdb', '')
            
            seed = model_path.split('seed_')[1].split("/")[0]  # seed-42-sample-0
            sample = model_path.split('sample_')[1].split(".")[0]  # seed-42-sample-0

            pdb = native_name
            
            
            rows.append({
                'method': method,
                'complex': native_name,
                'ref_pdb': pdb,
                'seed': seed,
                'id': sample,
                'dockq_score': dockq_score,
                'fnat': fnat,
                'lrmsd': lrmsd,
                'irmsd': irmsd,
            })
        except (ValueError, IndexError):
            continue
    
    return pd.DataFrame(rows) if rows else None

os.makedirs("results", exist_ok=True)
# 解析合并的DockQ输出并保存
for file in os.listdir("./"):
    if file.endswith(".out") and file.startswith("pred_v1_nomsa_notemplate"):
        method = file.split('pred_v1_')[1].split('.out')[0]
        df_all = parse_dockq_output(file)
        if df_all is not None:
            print(f'Total: {len(df_all)} rows, {df_all["method"].nunique()} methods')
            df_all.to_csv(f'./results/{method}_docking_results.csv', index=False)
            print(f'  {method}: {len(df_all)} rows, {df_all["complex"].nunique()} complexes')

else:
    print('No results found (run DockQ first)')

Total: 2550 rows, 1 methods
  nomsa_notemplate: 2550 rows, 170 complexes
No results found (run DockQ first)
